# 06. AI 음성 검색 시연 (STT + GPT 에이전트)

마이크 → 말하기 → 문장 인식(STT) → **AI 에이전트(GPT)가 추천 모델 입력값으로 해석** → 추천 5곳 + TMAP 지도.

| 단계 | 어디서 | 비용 |
|---|---|---|
| 음성 인식 (STT) | 브라우저 내장 (Web Speech API, 한국어) | 무료 |
| 문장 해석 (AI 에이전트) | OpenAI GPT (구조화 출력). 키가 없거나 실패하면 규칙 해석기로 대체 | 아주 적음 |
| 추천 | 내 PC에서 띄운 예측 서버 (`/agent`) + TMAP | TMAP 한도 안 |

에이전트가 하는 일: 출발지, 목적지, 출발 시각 또는 도착 목표("9시 회의"), 주차 시간, 선호("싼 데", "덜 걷게")를 뽑고,
음성 인식 오류를 고치고("스카이브" → "스카이뷰"), 목적지를 모르면 되묻고, "30분 늦게"처럼 이어 말해도 앞 대화를 반영한다.
크롬 또는 엣지에서 열어야 한다. 처음 열면 마이크, 위치 권한을 묻는다 (둘 다 허용).

## 1. 서버 라이브러리 확인 (처음 한 번)

없으면 설치한다. 설치했다면 Kernel → Restart Kernel 후 다음 셀부터.

In [ ]:
try:
    import fastapi, uvicorn
    print("fastapi", fastapi.__version__, "/ uvicorn", uvicorn.__version__, "준비됨")
except ImportError:
    %pip install fastapi uvicorn

## 2. GPT 키 저장 (처음 한 번)

[OpenAI API 키 페이지](https://platform.openai.com/api-keys)에서 만든 `sk-...` 키를 붙여 넣는다. 화면에는 안 보인다.
`서울시 공영주차장 예측/.env`에 TMAP 키와 함께 저장된다. 이미 저장했으면 건너뛴다.

In [ ]:
from getpass import getpass
from pathlib import Path

ENV = Path.home() / "서울시 공영주차장 예측" / ".env"
key = getpass("OpenAI API 키 붙여넣기 후 Enter: ").strip()
lines = [l for l in (ENV.read_text(encoding="utf-8").splitlines() if ENV.exists()) if not l.startswith("OPENAI_API_KEY=")]
ENV.write_text("\n".join(lines + [f"OPENAI_API_KEY={key}"]) + "\n", encoding="utf-8")
print("저장 완료:", ENV, "/ 키 길이:", len(key), "/ 들어 있는 키:", [l.split("=")[0] for l in lines + ["OPENAI_API_KEY="]])

## 3. AI 해석만 시험하기 (TMAP 호출 없음)

아무렇게나 말할 문장을 입력해서 에이전트가 어떻게 나누는지 본다. '해석: GPT'가 나오면 GPT 연결 성공이다.

In [ ]:
import sys
sys.path.insert(0, str(Path("..").resolve()))
from jariml.agent import understand
q = understand(input("문장: ") or "내일 아침 9시 회의인데 교대역 근처에 세울 데 있을까? 수원 SK스카이브에서 나가. 싼 데로")
print("해석:", q["source"], "|", q["understood"] or "", ("| 오류: " + q["error"]) if q["error"] else "")
print("출발지:", q["origin"] or "현재 위치", "/ 목적지:", q["dest"], "/",
      ("도착 목표 " + str(q["arrive_by"])) if q["arrive_by"] else ("출발 " + str(q["depart"] or "지금")),
      "/ 주차:", q["stay_min"], "/ 선호:", q["prefer"], "/ 되묻기:", q["question"])

## 4. 예측 서버 켜고 시연 페이지 열기

모델과 최신 주차 기록을 불러오느라 10~30초 걸린다. 브라우저에 시연 페이지가 새 탭으로 열린다.
코드나 키를 바꿨다면 5번으로 서버를 끄고 다시 켠다.

In [ ]:
import os, time, subprocess, webbrowser, urllib.request

ML = Path("..").resolve()
URL = "http://localhost:8000"

def alive():
    try:
        return urllib.request.urlopen(URL + "/health", timeout=2).status == 200
    except Exception:
        return False

if not alive():
    env = {**os.environ, "LOKY_MAX_CPU_COUNT": "4"}
    log = open(ML / "server.log", "w", encoding="utf-8")
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.app:app", "--port", "8000"],
                              cwd=ML, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(90):
        if alive() or server.poll() is not None:
            break
        time.sleep(1)
print("서버:", "켜짐" if alive() else "실패 (아래 로그 확인)")
if alive():
    webbrowser.open(URL + "/demo")
    print("시연 페이지:", URL + "/demo")
else:
    print((ML / "server.log").read_text(encoding="utf-8")[-2000:])

## 5. 서버 끄기

시연이 끝나거나 코드를 새로 받았으면 실행한다.

In [ ]:
try:
    server.terminate(); print("서버 꺼짐")
except NameError:
    print("이 노트북에서 켠 서버가 없다. 다른 곳에서 켰다면 Running 탭에서 그 노트북을 Shut Down한다.")